In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

# WRI vs RF Deprivation Statistics — Tables Notebook

## Purpose
Compile WRI Urban Land Use V1 per-country metrics into final population and segment summary tables
comparing WRI-based informality (`p_informal`) to CSMD RF deprivation labels across τ ∈ {0.1, 0.2, 0.3}.

This notebook compiles final WRI population tables from **preprocessed WRI per-block CSVs** deposited
on Zenodo. It does **not** require raw WRI rasters.

Raw WRI raster sampling (rasterio masking, `p_informal` computation per segment) is handled upstream in
`03_WRI_PerCountry_Metrics.ipynb`. That notebook produces per-block CSVs which are included in the
Zenodo deposit and used here.

## Data requirements

- **Preprocessed WRI per-block CSVs** (`data_external/zenodo/wri_per_country_outputs/`):
  Per-country output folders produced by `03_WRI_PerCountry_Metrics.ipynb` and deposited on Zenodo.
  This notebook reads `{country}_wri_vs_rf_per_block_with_preds.csv` from each country folder.
  Columns used: `ID_SEG`, `ID_HDC_G0`, `p_informal`, `rf_label`, `tif_name`.

- **RF prediction GPKGs** (`data_external/zenodo/predictions/`):
  Per-country `{country}_rf_preds.gpkg` files from the Zenodo deposit.
  Used here only to retrieve `POP_SEG` per segment via join on `(ID_HDC_G0, ID_SEG)`.

## Outputs (small; tracked in repository)

- `3_comparitive_analysis/WRI/Outputs/wri_rf_population_stats.csv`
- `3_comparitive_analysis/WRI/Outputs/wri_rf_population_summary_GLOBAL_rule_threshold_table_millions.csv`

## Reproducibility

This notebook is fully reproducible from the Zenodo deposit alone
(`wri_per_country_outputs/` + `predictions/`). No raw WRI rasters from GEE are required.

In [2]:
from pathlib import Path

# ============================================================
# PATH CONFIGURATION  (portable — no hard-coded local paths)
# ============================================================
REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "citysegmentdeprivation" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

DATA_EXTERNAL       = REPO_ROOT / "data_external"
ZENODO_DATA         = DATA_EXTERNAL / "zenodo"

PREDICTIONS_DIR     = ZENODO_DATA / "predictions"
WRI_PER_COUNTRY_DIR = ZENODO_DATA / "wri_per_country_outputs"
WRI_OUTPUT_DIR      = REPO_ROOT / "3_comparitive_analysis" / "WRI" / "Outputs"

WRI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output CSV paths (small, tracked in repository)
OUT_CSV = WRI_OUTPUT_DIR / "wri_rf_population_stats.csv"

# WRI threshold values
TAUS = [0.1, 0.2, 0.3]

print("WRI_PER_COUNTRY_DIR:", WRI_PER_COUNTRY_DIR.resolve())
print("PREDICTIONS_DIR    :", PREDICTIONS_DIR.resolve())
print("WRI_OUTPUT_DIR     :", WRI_OUTPUT_DIR.resolve())

WRI_PER_COUNTRY_DIR: F:\DEPRIMAP\DIRTY_MODEL\NATURE CITIES SUBMISSION FILES\cleangitrepo\citysegmentdeprivation\data_external\zenodo\wri_per_country_outputs
PREDICTIONS_DIR    : F:\DEPRIMAP\DIRTY_MODEL\NATURE CITIES SUBMISSION FILES\cleangitrepo\citysegmentdeprivation\data_external\zenodo\predictions
WRI_OUTPUT_DIR     : F:\DEPRIMAP\DIRTY_MODEL\NATURE CITIES SUBMISSION FILES\cleangitrepo\citysegmentdeprivation\3_comparitive_analysis\WRI\Outputs


In [3]:
# ------------------------------------------------------------------
# 2️⃣ HELPERS
# ------------------------------------------------------------------
def sort_for_export(df, preferred_cols):
    """Sort a DataFrame by available preferred columns for deterministic CSV output."""
    cols = [c for c in preferred_cols if c in df.columns]
    return df.sort_values(cols).reset_index(drop=True) if cols else df.reset_index(drop=True)

In [4]:
# ------------------------------------------------------------------
# 3️⃣ PER-COUNTRY PROCESSOR
#    Reads preprocessed Zenodo CSVs — no raw WRI rasters required.
# ------------------------------------------------------------------
def process_country(country):
    wri_path = WRI_PER_COUNTRY_DIR / country / f"{country}_wri_vs_rf_per_block_with_preds.csv"
    rf_path  = PREDICTIONS_DIR / f"{country}_rf_preds.gpkg"

    if not wri_path.exists():
        return {"country": country, "status": "missing_wri_csv"}
    if not rf_path.exists():
        return {"country": country, "status": "missing_rf_gpkg"}

    # ---- Read WRI per-block CSV ----
    try:
        wri_df = pd.read_csv(wri_path)
    except Exception as e:
        return {"country": country, "status": f"wri_read_error: {e}"}

    if "ID_SEG" not in wri_df.columns or "p_informal" not in wri_df.columns:
        return {"country": country, "status": "missing_required_cols_in_wri"}

    # Drop exact duplicate rows to prevent double-counting segments observed more than once
    # (e.g., from overlapping GEE export tiles). ID_SEG resets per city within a country,
    # so include ID_HDC_G0 (urban-area ID) to avoid incorrectly dropping valid cross-city rows.
    if "tif_name" in wri_df.columns and "ID_HDC_G0" in wri_df.columns:
        dedup_cols = ["tif_name", "ID_HDC_G0", "ID_SEG"]
    elif "tif_name" in wri_df.columns:
        dedup_cols = ["tif_name", "ID_SEG"]
    else:
        dedup_cols = ["ID_SEG"]
    wri_df = wri_df.drop_duplicates(subset=dedup_cols)

    # ---- Read RF GPKG for POP_SEG ----
    try:
        rf_gdf = gpd.read_file(rf_path)
    except Exception as e:
        return {"country": country, "status": f"gpkg_read_error: {e}"}

    if "ID_SEG" not in rf_gdf.columns:
        return {"country": country, "status": "missing_ID_SEG_in_gpkg"}
    if "POP_SEG" not in rf_gdf.columns:
        return {"country": country, "status": "missing_POP_SEG_in_gpkg"}

    # ---- Join: recover POP_SEG from RF GPKG ----
    # ID_SEG alone is not globally unique (it resets per city within a country).
    # Use (ID_HDC_G0, ID_SEG) when available for an unambiguous join key.
    if "ID_HDC_G0" in rf_gdf.columns and "ID_HDC_G0" in wri_df.columns:
        join_cols = ["ID_HDC_G0", "ID_SEG"]
    else:
        join_cols = ["ID_SEG"]

    rf_pop = rf_gdf[join_cols + ["POP_SEG"]].drop_duplicates(subset=join_cols)
    merged = wri_df.merge(rf_pop, on=join_cols, how="left")

    # Warn if population is missing for some rows after join
    missing_pop = int(merged["POP_SEG"].isna().sum())
    if missing_pop > 0:
        print(f"⚠️  {country}: {missing_pop}/{len(merged)} rows missing POP_SEG after join")

    # ---- Clean and type-cast ----
    merged = merged.dropna(subset=["rf_label", "p_informal", "POP_SEG"])
    merged["rf_label"]   = pd.to_numeric(merged["rf_label"],   errors="coerce")
    merged["p_informal"] = pd.to_numeric(merged["p_informal"], errors="coerce")
    merged["POP_SEG"]    = pd.to_numeric(merged["POP_SEG"],    errors="coerce")
    merged = merged.dropna(subset=["rf_label", "p_informal", "POP_SEG"])

    if merged.empty:
        return {"country": country, "status": "empty_after_join"}

    total_segments = len(merged)
    total_pop      = float(merged["POP_SEG"].sum())
    rf_dep_seg     = int((merged["rf_label"] == 1).sum())
    rf_dep_pop     = float(merged.loc[merged["rf_label"] == 1, "POP_SEG"].sum())

    rows = []
    for tau in TAUS:
        wri_dep = merged["p_informal"] >= tau
        rows.append({
            "country":                country,
            "threshold":              tau,
            "total_segments":         total_segments,
            "rf_deprived_segments":   rf_dep_seg,
            "wri_deprived_segments":  int(wri_dep.sum()),
            "total_population":       total_pop,
            "rf_deprived_population": rf_dep_pop,
            "wri_deprived_population":float(merged.loc[wri_dep, "POP_SEG"].sum()),
        })
    return rows

In [5]:
# ------------------------------------------------------------------
# 4️⃣ RUN ALL COUNTRIES
# ------------------------------------------------------------------
countries = sorted([p.name for p in WRI_PER_COUNTRY_DIR.iterdir() if p.is_dir()])
print(f"Found {len(countries)} country folders in wri_per_country_outputs/\n")

all_rows = []
skipped  = []

for country in countries:
    result = process_country(country)
    if isinstance(result, list):
        all_rows.extend(result)
    else:
        skipped.append(result)
        print(f"  ⚠️  Skipped {country}: {result.get('status', 'unknown')}")

df = pd.DataFrame(all_rows)

print(f"\n✅ Processed {len(countries) - len(skipped)} / {len(countries)} countries successfully")
if skipped:
    print(f"   Skipped ({len(skipped)}): {[s['country'] for s in skipped]}")
print(f"   Total per-country rows: {len(df)}  ({len(df) // len(TAUS) if TAUS else '?'} countries × {len(TAUS)} thresholds)")
df.head()

Found 48 country folders in wri_per_country_outputs/




✅ Processed 48 / 48 countries successfully
   Total per-country rows: 144  (48 countries × 3 thresholds)


,country,threshold,total_segments,rf_deprived_segments,wri_deprived_segments,total_population,rf_deprived_population,wri_deprived_population
0,afghanistan,0.1,4073,154,3134,6.114890e+06,446600.874311,4.492671e+06
1,afghanistan,0.2,4073,154,3006,6.114890e+06,446600.874311,4.052900e+06
2,afghanistan,0.3,4073,154,2889,6.114890e+06,446600.874311,3.778005e+06
3,algeria,0.1,2479,588,935,4.424830e+06,635186.354558,1.557636e+06
4,algeria,0.2,2479,588,716,4.424830e+06,635186.354558,1.098852e+06


In [6]:
# ------------------------------------------------------------------
# 5️⃣ ADD GLOBAL SUMMARY ROW
# ------------------------------------------------------------------
global_rows = []
for tau in TAUS:
    sub = df[df["threshold"] == tau]
    global_rows.append({
        "country":                "GLOBAL",
        "threshold":              tau,
        "total_segments":         sub["total_segments"].sum(),
        "rf_deprived_segments":   sub["rf_deprived_segments"].sum(),
        "wri_deprived_segments":  sub["wri_deprived_segments"].sum(),
        "total_population":       sub["total_population"].sum(),
        "rf_deprived_population": sub["rf_deprived_population"].sum(),
        "wri_deprived_population":sub["wri_deprived_population"].sum(),
    })
global_df = pd.DataFrame(global_rows)

final = pd.concat([df, global_df], ignore_index=True)

In [7]:
# ------------------------------------------------------------------
# 6️⃣ SAVE OUTPUT
# ------------------------------------------------------------------
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
sort_for_export(final, ["country", "threshold"]).to_csv(OUT_CSV, index=False)
print(f"✅ Saved {len(final)} rows to:\n   {OUT_CSV}")

print("\nSample output (first 12 rows):")
try:
    from IPython.display import display
    display(final.head(12))
except Exception:
    print(final.head(12).to_string())

✅ Saved 147 rows to:
   F:\DEPRIMAP\DIRTY_MODEL\NATURE CITIES SUBMISSION FILES\cleangitrepo\citysegmentdeprivation\3_comparitive_analysis\WRI\Outputs\wri_rf_population_stats.csv

Sample output (first 12 rows):


,country,threshold,total_segments,rf_deprived_segments,wri_deprived_segments,total_population,rf_deprived_population,wri_deprived_population
0,afghanistan,0.1,4073,154,3134,6.114890e+06,4.466009e+05,4.492671e+06
1,afghanistan,0.2,4073,154,3006,6.114890e+06,4.466009e+05,4.052900e+06
2,afghanistan,0.3,4073,154,2889,6.114890e+06,4.466009e+05,3.778005e+06
3,algeria,0.1,2479,588,935,4.424830e+06,6.351864e+05,1.557636e+06
4,algeria,0.2,2479,588,716,4.424830e+06,6.351864e+05,1.098852e+06
5,algeria,0.3,2479,588,558,4.424830e+06,6.351864e+05,7.361825e+05
6,angola,0.1,9106,6364,8048,1.078201e+07,7.043579e+06,9.139495e+06
7,angola,0.2,9106,6364,7565,1.078201e+07,7.043579e+06,8.421709e+06
8,angola,0.3,9106,6364,7095,1.078201e+07,7.043579e+06,7.718899e+06
9,argentina,0.1,17130,1276,5032,1.545089e+07,1.145545e+06,4.618489e+06


In [8]:
# ============================================================
# 7️⃣ Global Summary Table — WRI vs RF (p_informal, in millions)
# ============================================================

# --- Paths ---
IN_CSV  = WRI_OUTPUT_DIR / "wri_rf_population_stats.csv"
OUT_CSV_GLOBAL = WRI_OUTPUT_DIR / "wri_rf_population_summary_GLOBAL_rule_threshold_table_millions.csv"

# --- Load data ---
df_in = pd.read_csv(IN_CSV)

# --- Keep only needed columns ---
keep_cols = [
    "country",
    "threshold",
    "total_segments",
    "rf_deprived_segments",
    "wri_deprived_segments",
    "total_population",
    "rf_deprived_population",
    "wri_deprived_population",
]
df_in = df_in[keep_cols].copy()

# --- Global aggregation (sum across countries, excluding GLOBAL row to avoid double-count) ---
df_countries = df_in[df_in["country"] != "GLOBAL"].copy()
global_summary = (
    df_countries.groupby("threshold", as_index=False)
                .agg({
                    "total_segments":         "sum",
                    "rf_deprived_segments":   "sum",
                    "wri_deprived_segments":  "sum",
                    "total_population":       "sum",
                    "rf_deprived_population": "sum",
                    "wri_deprived_population":"sum",
                })
)

# --- Convert populations to millions and calculate shares ---
summary_rows = []
for _, row in global_summary.iterrows():
    summary_rows.append({
        "Rule":                  "WRI (p_informal)",
        "Threshold":             row["threshold"],
        "TotalSegments":         int(row["total_segments"]),
        "RF_Deprived_Seg":       int(row["rf_deprived_segments"]),
        "Rule_Deprived_Seg":     int(row["wri_deprived_segments"]),
        "Total_Pop_M":           round(row["total_population"]        / 1e6, 2),
        "RF_Deprived_Pop_M":     round(row["rf_deprived_population"]  / 1e6, 2),
        "Rule_Deprived_Pop_M":   round(row["wri_deprived_population"] / 1e6, 2),
    })

out = sort_for_export(pd.DataFrame(summary_rows), ["Rule", "Threshold"])

# --- Add % shares ---
out["RF_Deprived_Pop_%"]   = (out["RF_Deprived_Pop_M"]   / out["Total_Pop_M"] * 100).round(2)
out["Rule_Deprived_Pop_%"] = (out["Rule_Deprived_Pop_M"] / out["Total_Pop_M"] * 100).round(2)

# --- Save and preview ---
out.to_csv(OUT_CSV_GLOBAL, index=False)

print("\n✅ Global WRI–RF Summary Table (Millions):\n")
print(out.to_string(index=False))
print(f"\n✅ Saved to:\n   {OUT_CSV_GLOBAL}")


✅ Global WRI–RF Summary Table (Millions):

            Rule  Threshold  TotalSegments  RF_Deprived_Seg  Rule_Deprived_Seg  Total_Pop_M  RF_Deprived_Pop_M  Rule_Deprived_Pop_M  RF_Deprived_Pop_%  Rule_Deprived_Pop_%
WRI (p_informal)        0.1         358104            92950             154785       525.63             128.35               227.18              24.42                43.22
WRI (p_informal)        0.2         358104            92950             137519       525.63             128.35               186.10              24.42                35.41
WRI (p_informal)        0.3         358104            92950             124501       525.63             128.35               159.71              24.42                30.38

✅ Saved to:
   F:\DEPRIMAP\DIRTY_MODEL\NATURE CITIES SUBMISSION FILES\cleangitrepo\citysegmentdeprivation\3_comparitive_analysis\WRI\Outputs\wri_rf_population_summary_GLOBAL_rule_threshold_table_millions.csv
